# 🧩 WE7 · Notebook 02 — One backbone, many tasks
## Adapting a pretrained model a hundred times without storing it a hundred times

> **The situation.** You own one pretrained model and a queue of people who all want the same
> thing: *that* model, adapted to *their* task. One group has spectra, one has scanned forms, one
> has a client-specific classifier, and by the end of the quarter there will be a hundred of them.
>
> The obvious plan is to fine-tune the model once per task and save the result. That plan has a
> price tag, and the price is not the training: it is that every task now owns a complete copy of
> a model that is mostly identical to every other copy. A hundred tasks, a hundred full
> checkpoints, a hundred training runs that each have to hold gradients and optimiser state for
> every parameter in the model.
>
> **Parameter-efficient fine-tuning (PEFT)** is the family of answers to that. The idea in one
> line: freeze the pretrained model, add or select a *small* set of parameters, train only those,
> and ship only those.

**What you will be able to do by the end**

- tell **total parameters** from **trainable parameters**, and say why the second number is the
  one that drives gradient and optimiser memory;
- implement **BitFit**, **LoRA** and **Diff Pruning** on the same pretrained model, from
  scratch, in a few lines each;
- work out a LoRA update's parameter count *before* running anything;
- compare methods on accuracy, memory, checkpoint size and runtime at once, and argue for one.

**How this notebook works**
- Short explanations, then small hands-on tasks marked **🎯** for you to fill in.
- One dashboard is redrawn after every method, so the comparison builds up in front of you.
- Everything runs on **CPU** in well under a minute of compute. No GPU, no downloads, no
  Transformers. The model is a 4,482-parameter MLP you can print in full.
- ⏱️ About an hour.

> 🧠 **What you need to know already:** what a linear layer is, what a training loop does, and
> that gradients get computed and applied. Nothing about Transformers, attention or language
> models is assumed anywhere in this notebook.

## 0. Setup

This notebook is **self-contained**: the first cell pulls the exercise files (the `peft_viz.py`
display helpers) directly from the course repository. Run the setup cells below in order.

> 💡 The setup cells clone the course repo into your Colab session and install the (CPU-only)
> dependencies. Nothing to install by hand, no account and no access token needed. Outside
> Colab it works too: the cell finds the repo root on disk instead of cloning.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys, subprocess

REPO_OWNER  = "eth-fdd-fs26"
REPO_NAME   = "FDD-WE7-public"
REPO_BRANCH = "main"
HELPER      = os.path.join("2_peft_finetuning", "exercise", "peft_viz.py")

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _git(args):
    '''Run git without a terminal prompt: the repo is public, so no credentials are needed.'''
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    done = subprocess.run(args, capture_output=True, text=True, env=env)
    return done.returncode, (done.stdout + done.stderr).strip()

if _in_colab():
    url = "https://github.com/%s/%s.git" % (REPO_OWNER, REPO_NAME)

    if os.path.isdir(REPO_NAME):
        print("Updating the exercise repo to the latest version...")
        code, log = _git(["git", "-C", REPO_NAME, "pull", "-q", url, REPO_BRANCH])
    else:
        print("Cloning the exercise repo...")
        code, log = _git(["git", "clone", "-q", "-b", REPO_BRANCH, url, REPO_NAME])

    if code != 0:
        print(log)
        raise RuntimeError(
            "git failed. Check that you are online and run this cell again. If it keeps "
            "failing, delete the %r folder in the file browser (left sidebar) and retry."
            % REPO_NAME)

# Move to the REPO ROOT - the folder holding `2_peft_finetuning/exercise/` - so imports resolve.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()),
              os.path.dirname(os.path.dirname(os.getcwd())), os.getcwd()]:
    if os.path.exists(os.path.join(_root, HELPER)):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Cloned, but %s is not there - the clone did not bring the exercise files." % HELPER)
sys.path.insert(0, os.path.join(os.getcwd(), "2_peft_finetuning", "exercise"))
print("Working directory:", os.getcwd())

**0.2 — Install dependencies.** All of these are already on Colab; this just pins versions
(and makes the notebook work outside Colab too). The CPU build of PyTorch is all we need.

In [ ]:
%pip install -q -r 2_peft_finetuning/exercise/requirements_peft.txt

**0.3 — Import the libraries.** The dashboards, diagrams and quizzes live in **`peft_viz`**
so the teaching cells stay about the *idea* rather than about HTML.

In [ ]:
import copy, time, os
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import make_moons

import importlib
import peft_viz as pv
importlib.reload(pv)      # pick up the latest helpers even if a stale copy was cached

torch.set_num_threads(1)  # so the measured times below mean something on any machine

SEED = 0

def seed_everything(seed=SEED):
    '''Every experiment in this notebook starts from the same random state.'''
    torch.manual_seed(seed)
    np.random.seed(seed)

seed_everything()
print("torch", torch.__version__, "· device: cpu · seeds fixed ✅")

---
# Part 0 — The bill for one model per task  ·  ~5 min

Start with the situation, not with the method.

You have **one pretrained model**. A hundred groups want it adapted to a hundred tasks. Each
adapted version has to be trained, stored, transferred and loaded. Before any of that, ask what
the storage alone costs.

Here is the picture the whole notebook keeps coming back to.

In [ ]:
pv.one_model_per_task(base_parameter_count=1_000_000_000, bytes_per_param=2, n_tasks=3)

Two numbers decide everything on that picture. Change them and see what happens.

In [ ]:
base_parameter_count = 1_000_000_000      # a 1B-parameter pretrained model
number_of_tasks      = 20                 # how many adapted versions you have to keep

pv.storage_scenario(base_parameter_count, number_of_tasks,
                    module_fraction=0.005,    # a task module of 0.5% of the base model
                    bytes_per_param=2)        # FP16

> 📏 **Units, once.** Throughout this notebook 1 KB = 1000 bytes, 1 MB = 10⁶ bytes, and so on,
> which is how disks are sold and how the arithmetic below comes out round.

### 🎯 Task — do the arithmetic yourself

A **1-billion-parameter model in FP16** (2 bytes per parameter), and **20 tasks**. Work out both
totals: twenty complete copies, then one shared backbone plus twenty modules of 0.5% each.

<details><summary>💡 <b>Hint 1</b> — what are we actually asking?</summary>

Storage is *number of parameters x bytes per parameter*. For the full-copy plan, every task owns a complete model. For the PEFT plan, the backbone is stored exactly once no matter how many tasks there are, and only the module repeats.

</details>

<details><summary>🔧 <b>Hint 2</b> — which variable, which function</summary>

One copy is `base_parameter_count * bytes_per_param`. A module is `module_fraction` of that. The PEFT total is one backbone plus `number_of_tasks` modules.

</details>

In [ ]:
base_parameter_count = 1_000_000_000
bytes_per_param      = 2            # FP16
number_of_tasks      = 20
module_fraction      = 0.005        # 0.5% of the base model

bytes_per_full_copy = ???           # 🎯 one complete fine-tuned model
bytes_per_module    = ???           # 🎯 one task-specific module

full_finetuning_total = ???         # 🎯 one complete copy per task
peft_total            = ???         # 🎯 the backbone once, plus one module per task

print("one full copy   :", bytes_per_full_copy / 1e9, "GB")
print("one task module :", bytes_per_module / 1e6, "MB")
print("20 full copies  :", full_finetuning_total / 1e9, "GB")
print("shared + modules:", peft_total / 1e9, "GB")
print("ratio           : {:.1f}x less storage".format(full_finetuning_total / peft_total))

Check the three numbers against your own arithmetic:

In [ ]:
pv.number_quiz("storage_math")

Notice what the second plan did **not** do: it did not make the model smaller. The backbone
is the same size it always was, it still has to be loaded, and it still runs on every forward
pass. What collapsed is the cost of *task number 21*.

### 🧠 Quick check

In [ ]:
pv.mc_quiz("why_peft")

> ### What you just did
> You priced the problem before solving it. The rest of the notebook is about the other half of
> the bill: what a training run has to hold in memory while it produces one of those task
> modules. From here on the model is small enough to print, so every number is checkable.

---
# Part 1 — One shared backbone, pretrained  ·  ~7 min

We need a pretrained model and a task to adapt it to. Both are two-dimensional, so you can see
every decision the model makes.

**The source task** is the classic two-moons problem. **The target task** is the same shape,
rotated and shifted, with more noise: related enough that the pretrained model is worth keeping,
different enough that it has to be adapted.

> This toy setup stands in for the real one on purpose. A 4,482-parameter MLP and a 7-billion
> parameter language model pose the same systems question (what has to be trained, what has to be
> stored per task), and only the small one fits on a slide.

In [ ]:
def make_task(rotate_deg=0.0, shift=(0.0, 0.0), noise=0.20, n=600, seed=0):
    '''Two moons, optionally rotated and shifted. Same generator for both tasks.'''
    X, y = make_moons(n_samples=n, noise=noise, random_state=seed)
    t = np.radians(rotate_deg)
    R = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
    X = X @ R.T + np.array(shift)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

def split(X, y, n_train=400):
    return X[:n_train], y[:n_train], X[n_train:], y[n_train:]

# SOURCE task - what the backbone is pretrained on
X_src, y_src = make_task(rotate_deg=0.0, shift=(0.0, 0.0), noise=0.20, seed=0)
X_src_train, y_src_train, X_src_test, y_src_test = split(X_src, y_src)

# TARGET task - what every method below has to adapt to
X_tgt, y_tgt = make_task(rotate_deg=75.0, shift=(0.6, -0.5), noise=0.25, seed=1)
X_tgt_train, y_tgt_train, X_tgt_test, y_tgt_test = split(X_tgt, y_tgt)

print("source: %d train / %d test" % (len(X_src_train), len(X_src_test)))
print("target: %d train / %d test" % (len(X_tgt_train), len(X_tgt_test)))
pv.show_tasks(X_src, y_src, X_tgt, y_tgt)

## 1.1 · The model

Three linear layers with ReLU between them. That is the entire architecture.

```
2 inputs  ->  Linear(2, 64)  ->  ReLU  ->  Linear(64, 64)  ->  ReLU  ->  Linear(64, 2)
```

A `Linear(a, b)` layer holds a **weight matrix** of shape `(b, a)` and a **bias vector** of
length `b`. A forward pass multiplies by the matrix, adds the bias, and passes the result on.
That is all you need to follow every method in this notebook.

In [ ]:
HIDDEN = 64

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, HIDDEN)
        self.fc2 = nn.Linear(HIDDEN, HIDDEN)
        self.fc3 = nn.Linear(HIDDEN, 2)      # the classification head

    def forward(self, x):
        h1 = torch.relu(self.fc1(x))
        h2 = torch.relu(self.fc2(h1))
        return self.fc3(h2)

seed_everything()
demo = MLP()
print(demo)

### 🎯 Task — count a layer before PyTorch tells you

`fc1` is `nn.Linear(2, 64)`. Work out its weights and biases on paper first (no code), then we
verify.

In [ ]:
pv.number_quiz("layer_math")

Now the verification. Every parameter tensor has a `.shape` and a `.numel()` (number of
elements), and a `.requires_grad` flag saying whether a gradient will be computed for it. Look at
one layer first.

In [ ]:
for name, p in demo.fc1.named_parameters():
    print("%-8s shape %-12s %4d elements   requires_grad=%s"
          % (name, tuple(p.shape), p.numel(), p.requires_grad))

Same loop over the whole model, plus the totals. This little function gets used after every
method, so read it once now.

In [ ]:
def describe(model):
    '''Every parameter tensor: name, shape, size, and whether it is being trained.'''
    total, trainable = 0, 0
    for name, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
        print("%-16s %-14s %6d  %s"
              % (name, tuple(p.shape), p.numel(),
                 "TRAINABLE" if p.requires_grad else "frozen"))
    print("-" * 52)
    print("total     : %6d" % total)
    print("trainable : %6d  (%.2f%% of the model)"
          % (trainable, 100 * trainable / total))

describe(demo)

### 🎯 Task — the two counts, in code

These two expressions come back in every part of the notebook, so write them once by hand.

<details><summary>💡 <b>Hint 1</b> — what are we actually asking?</summary>

`model.parameters()` yields the tensors. `.numel()` gives the number of elements in one tensor. The trainable count is the same sum, restricted to the tensors that carry a gradient.

</details>

<details><summary>🔧 <b>Hint 2</b> — which variable, which function</summary>

`sum(p.numel() for p in model.parameters())` for the first. For the second, filter on `p.requires_grad`.

</details>

In [ ]:
def count_parameters(model):
    return ???        # 🎯 every parameter the forward pass needs

def count_trainable(model):
    return ???        # 🎯 only the ones that get a gradient and an update

print("total    :", count_parameters(demo))
print("trainable:", count_trainable(demo))
assert count_parameters(demo) == 4482, "192 + 4160 + 130 = 4482"
print("✅ 4,482 parameters, of which the middle layer alone is 4,160")

## 1.2 · Pretrain it, once

Everything else in this notebook starts from the state produced by this cell, so it runs once and
is never touched again. The training loop is the one you already know: full batch, Adam, cross
entropy.

In [ ]:
def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

def train(model, X, y, epochs=200, lr=0.01):
    '''Train whatever has requires_grad=True, and report how long it took.'''
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimiser = torch.optim.Adam(trainable, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    start = time.perf_counter()
    for _ in range(epochs):
        optimiser.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimiser.step()
    return time.perf_counter() - start

seed_everything()
backbone = MLP()
pretrain_time = train(backbone, X_src_train, y_src_train, epochs=400, lr=0.01)

PRETRAINED_STATE = copy.deepcopy(backbone.state_dict())   # the one shared starting point

acc_src_before = accuracy(backbone, X_src_test, y_src_test)
acc_tgt_before = accuracy(backbone, X_tgt_test, y_tgt_test)
print("pretraining took %.2f s" % pretrain_time)
print("source accuracy : %.1f%%" % (100 * acc_src_before))
print("target accuracy : %.1f%%   <- this is what every method below has to fix"
      % (100 * acc_tgt_before))

The boundary the pretrained model draws, on both tasks. It fits the one it was trained on
and cuts straight through the other.

In [ ]:
def predict_fn(model):
    '''A function the plotter can call on a grid of points.'''
    def f(grid):
        model.eval()
        with torch.no_grad():
            return model(torch.tensor(grid)).argmax(dim=1).numpy()
    return f

pv.boundaries([("pretrained · source task", predict_fn(backbone))], X_src, y_src, ncols=1)
pv.boundaries([("pretrained · target task", predict_fn(backbone))], X_tgt, y_tgt, ncols=1)

And here is the same model as a map. Every block is a parameter tensor, and right now all
of them are trainable.

In [ ]:
def map_entries(model):
    return [(n, p.numel(), p.requires_grad) for n, p in model.named_parameters()]

pv.model_map(map_entries(backbone), title="The pretrained backbone, nothing frozen yet")

> ### What you just did
> You have a pretrained model, a target task it fails on, and two functions that count
> parameters. Everything from here on starts from `PRETRAINED_STATE`: the *identical* pretrained
> weights, every time, so the comparison at the end is fair.

---
# Part 2 — Full fine-tuning, the baseline  ·  ~7 min

The obvious method first: take the pretrained model, unfreeze everything, train it on the target
task. This is the yardstick. Every later method is reported as a percentage of it.

Three quantities get measured for every method, and they are **not** the same thing:

| quantity | what it means |
|---|---|
| **training state** | parameters + gradients + optimiser state, held in memory *while training* |
| **task checkpoint** | the bytes you have to save on disk *for this one task* |
| **runtime** | the work done per forward and backward pass |

A method can be excellent on one and unremarkable on another. Keeping the three apart is most of
the point of this notebook.

## 2.1 · The training-state estimate

Assume FP32 (4 bytes per number) and an Adam-style optimiser, which keeps **two** running moments
per optimised parameter. Then, per parameter:

- stored value: **4 bytes**, for every parameter, frozen or not;
- gradient: **4 bytes**, for trainable parameters only;
- optimiser moments: **8 bytes** (2 x 4), for trainable parameters only.

$$M_{\text{estimated}} = 4\,N_{\text{total}} + 4\,N_{\text{trainable}} + 8\,N_{\text{trainable}}$$

> ⚠️ **This is a simplified estimate, not a measurement.** It ignores activations, temporary
> buffers, allocator overhead and framework copies. Activations matter in practice: the frozen
> backbone still has to run, and enough of it has to be kept around to push gradients back to
> whatever *is* trainable. Never claim that PEFT cuts total training memory in proportion to the
> percentage of trainable parameters.

### 🎯 Task — write the formula

<details><summary>💡 <b>Hint 1</b> — what are we actually asking?</summary>

Three terms. The first one covers the whole model because every parameter has to be stored to run the forward pass, frozen or not. The other two only apply to parameters the optimiser touches.

</details>

<details><summary>🔧 <b>Hint 2</b> — which variable, which function</summary>

`4 * n_total + 4 * n_trainable + 8 * n_trainable`, with the 8 coming from Adam's two moments at 4 bytes each.

</details>

In [ ]:
def training_state_bytes(n_total, n_trainable, bytes_per_param=4):
    weights   = ???      # 🎯 every parameter has to be stored
    gradients = ???      # 🎯 one gradient per trainable parameter
    optimiser = ???      # 🎯 Adam keeps TWO moments per trainable parameter
    return weights + gradients + optimiser

# a quick sanity check on the model you already have
print("all 4482 trainable :", training_state_bytes(4482, 4482), "bytes")
print("only 130 trainable :", training_state_bytes(4482, 130), "bytes")
assert training_state_bytes(4482, 4482) == 4482 * 16
assert training_state_bytes(4482, 130) == 4482 * 4 + 130 * 12
print("✅ freezing shrinks two of the three terms, and leaves the first one alone")

## 2.2 · The rest of the measuring kit

Checkpoints are written to a real file and measured with `os.path.getsize`, so nothing here is
hypothetical. Inference timing gets a warm-up and many repetitions, because a single forward pass
of a model this small is mostly noise.

In [ ]:
import tempfile
CKPT_DIR = tempfile.mkdtemp(prefix="peft_ckpt_")

def checkpoint_bytes(tensors, name):
    '''Save exactly these tensors, then ask the filesystem how big the file is.'''
    path = os.path.join(CKPT_DIR, name + ".pt")
    torch.save(tensors, path)
    return os.path.getsize(path)

def inference_ms(model, X, repeats=200, warmup=20):
    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            model(X)
        start = time.perf_counter()
        for _ in range(repeats):
            model(X)
    return (time.perf_counter() - start) / repeats * 1000

def measure(method, model, train_time, ckpt, trainable_override=None,
            train_state_override=None, **flags):
    '''One row of the comparison. Nothing is estimated except the memory figure.

    The two overrides exist for one method only (Diff Pruning in Part 5), where
    what is trained and what is served are not the same object.
    '''
    total = count_parameters(model)
    trainable = trainable_override or count_trainable(model)
    record = {
        "method": method,
        "total_params": total,
        "trainable_params": trainable,
        "train_state_bytes": train_state_override or training_state_bytes(total, trainable),
        "ckpt_bytes": ckpt,
        "target_acc": accuracy(model, X_tgt_test, y_tgt_test),
        "source_acc": accuracy(model, X_src_test, y_src_test),
        "train_time_s": train_time,
        "infer_ms": inference_ms(model, X_tgt_test),
    }
    record.update(flags)
    return record

results = []      # every method appends one record here
print("checkpoints will be written to", CKPT_DIR)

## 2.3 · Fine-tune everything

Note two habits worth keeping even when they look redundant. The model is rebuilt from
`PRETRAINED_STATE` rather than reused, and the optimiser is constructed from the parameters that
have `requires_grad=True`, even though here that is all of them. Both make the later methods a
one-line change instead of a rewrite.

In [ ]:
def fresh_backbone():
    '''An exact copy of the pretrained model. Every method starts here.'''
    model = MLP()
    model.load_state_dict(copy.deepcopy(PRETRAINED_STATE))
    return model

seed_everything()
full_model = fresh_backbone()

for p in full_model.parameters():
    p.requires_grad = True                      # the baseline trains everything

total, trainable = count_parameters(full_model), count_trainable(full_model)
print("total %d · trainable %d (%.1f%%)" % (total, trainable, 100 * trainable / total))

full_time = train(full_model, X_tgt_train, y_tgt_train, epochs=200, lr=0.01)
print("trained in %.2f s" % full_time)
print("target accuracy: %.1f%%  (was %.1f%% before adapting)"
      % (100 * accuracy(full_model, X_tgt_test, y_tgt_test), 100 * acc_tgt_before))

The checkpoint for this task is the whole model: every parameter changed, so every
parameter has to be saved.

In [ ]:
full_ckpt = checkpoint_bytes(full_model.state_dict(), "full_finetuning")
print("task checkpoint:", full_ckpt, "bytes")

full_record = measure("Full fine-tuning", full_model, full_time, full_ckpt,
                      changes_path=False, mergeable=False, shares_backbone=False,
                      advantage="Every parameter is free to move",
                      limitation="A complete training state and a complete checkpoint per task")
results.append(full_record)
BASELINE = full_record          # every later dashboard is a percentage of this one

pv.dashboard(full_record)

Two views of the same run. First, which parameters were trainable:

In [ ]:
pv.param_strip(full_record["total_params"], full_record["trainable_params"],
               title="Full fine-tuning · nothing is frozen")

And where the training-state estimate goes. Three quarters of it is there only because we
are optimising every parameter: the gradients and the two Adam moments.

In [ ]:
pv.memory_stack(full_record["total_params"], full_record["trainable_params"])

### 🧠 Quick check
Click every statement you think is **true**.

In [ ]:
pv.true_false_quiz("counting")

> ### What you just did
> You have the yardstick: a full training state, a full checkpoint, and the accuracy that comes
> from letting every parameter move. Everything from here is an attempt to keep most of that
> accuracy while moving far fewer of them.

---
# Part 3 — BitFit: train the parameters you already have  ·  ~6 min

First PEFT method, and the smallest possible one:

> ### **BitFit**: freeze every weight matrix, and train only the bias vectors the model already
> ### contains.

Nothing is added to the model. No new layer, no new tensor, no change to the forward pass. The
only thing that changes is which parameters carry a gradient.

> 🧷 **The rule for every PEFT method in this notebook.** The classification head `fc3.weight`
> stays **frozen**, so that what adapts is the method's own mechanism and not a retrained output
> layer. BitFit is the one place where the head's *bias* moves, because a bias is exactly what
> BitFit trains. That policy is stated here, applies to every method below, and is visible in
> every parameter listing.

## 3.1 · Freeze everything, then let the biases back in

Two steps, deliberately separate: freeze the lot, then unfreeze what this method trains. Every
method below has the same shape.

### 🎯 Task — write the condition

<details><summary>💡 <b>Hint 1</b> — what are we actually asking?</summary>

PyTorch names parameters `<layer>.<kind>`, so a bias is a parameter whose name ends in `bias`. You want to set the flag on those, and leave the frozen ones alone.

</details>

<details><summary>🔧 <b>Hint 2</b> — which variable, which function</summary>

`name.endswith("bias")`. The loop variable is `name`, and the flag is `p.requires_grad`.

</details>

In [ ]:
seed_everything()
bitfit_model = fresh_backbone()

for p in bitfit_model.parameters():          # step 1: freeze the entire model
    p.requires_grad = False

for name, p in bitfit_model.named_parameters():
    if ???:                                  # 🎯 which parameters does BitFit train?
        p.requires_grad = True

describe(bitfit_model)

64 + 64 + 2 = 130 numbers, out of 4,482. Here is the same fact as a map: the weight
matrices stay grey, the thin bias strips turn orange.

In [ ]:
pv.model_map(map_entries(bitfit_model),
             title="BitFit · the weight matrices never move")

## 3.2 · Train and measure

Same data, same optimiser, same epochs, same learning rate as the baseline. The only difference
is the set of parameters handed to Adam.

In [ ]:
bitfit_time = train(bitfit_model, X_tgt_train, y_tgt_train, epochs=200, lr=0.01)
print("trained in %.2f s  (full fine-tuning took %.2f s)" % (bitfit_time, full_time))

### 🎯 Task — save only what this task owns

A PEFT checkpoint holds the trainable parameters and nothing else. The frozen backbone is already
on disk, once, shared by every task.

<details><summary>💡 <b>Hint 1</b> — what are we actually asking?</summary>

You want a dictionary mapping parameter names to tensors, containing only the parameters that are being trained. Everything else is in the shared backbone file.

</details>

<details><summary>🔧 <b>Hint 2</b> — which variable, which function</summary>

Iterate over `model.named_parameters()` and keep the ones where `p.requires_grad` is True. A dict comprehension does it in one line.

</details>

In [ ]:
def task_checkpoint(model):
    '''Only the parameters this task actually changed.'''
    return {???}          # 🎯 name -> tensor, for the trainable parameters only

bitfit_state = task_checkpoint(bitfit_model)
bitfit_ckpt = checkpoint_bytes(bitfit_state, "bitfit")

print("saved tensors:", list(bitfit_state.keys()))
print("checkpoint: %d bytes  (full fine-tuning: %d bytes)" % (bitfit_ckpt, full_ckpt))

In [ ]:
bitfit_record = measure("BitFit", bitfit_model, bitfit_time, bitfit_ckpt,
                        changes_path=False, mergeable=True,
                        advantage="Nothing is added to the model at all",
                        limitation="Very little capacity: biases shift outputs, they cannot "
                                   "reshape what a layer computes")
results.append(bitfit_record)

pv.dashboard(bitfit_record, baseline=BASELINE)

Look at the four bars before reading on. Trainable parameters and checkpoint size fell off
a cliff. The training-state estimate fell much less, because the whole model still has to be
stored. And the training time moved far less than either of them: the frozen backbone still runs forwards
on every step, and gradients still travel back through it to reach the biases.

The boundary this bought:

In [ ]:
pv.boundaries([("pretrained", predict_fn(backbone)),
               ("full fine-tuning", predict_fn(full_model)),
               ("BitFit", predict_fn(bitfit_model))],
              X_tgt, y_tgt, title="Target task")

### 🧠 Quick check

In [ ]:
pv.mc_quiz("frozen_forward")

> ### What you just did
> You adapted a pretrained model by selecting 2.9% of the parameters it already had. That is one
> of the two families of PEFT methods: **select an existing subset**. The other family
> **adds something new** and trains that instead, which is where the next part goes. Watch what it
> buys you, and what BitFit could not do: reshape what a layer computes.

---
# Part 4 — LoRA: constrain the update, not the weight  ·  ~11 min

Forget the whole model for a moment and look at **one linear layer**:

$$y = Wx + b, \qquad W \in \mathbb{R}^{d_{out} \times d_{in}}$$

Fine-tuning that layer means learning a change $\Delta W$ and serving $W + \Delta W$. Full
fine-tuning learns every one of the $d_{out} \cdot d_{in}$ entries of $\Delta W$ independently.

**LoRA** (low-rank adaptation) makes one assumption: the *change* a new task needs is much
simpler than the weight matrix itself, so it can be written as a product of two thin matrices.

$$\Delta W = BA, \qquad A \in \mathbb{R}^{r \times d_{in}}, \quad B \in \mathbb{R}^{d_{out} \times r}$$

$$y = Wx + \frac{\alpha}{r}\,BAx + b$$

$W$ stays frozen and full-rank. Only $A$ and $B$ are trained, and the whole task update is
$r(d_{in} + d_{out})$ numbers instead of $d_{out} d_{in}$. The scalar $\alpha/r$ is a fixed
scaling that keeps the size of the update roughly comparable as you change $r$.

$B$ starts at **zero**, so $BA = 0$ and the adapted layer starts out identical to the pretrained
one. Training moves it away from there, rather than towards it from somewhere random.

Move the rank and watch the two thin matrices change shape.

In [ ]:
pv.lora_widget(d_in=HIDDEN, d_out=HIDDEN, ranks=(1, 2, 4, 8, 16), default=4)

### 🎯 Task — the cost, on paper

The layer we will attach it to is `fc2`, with $d_{in} = 64$ and $d_{out} = 64$, at rank
$r = 4$.

In [ ]:
pv.number_quiz("lora_math")

## 4.1 · The module

We attach LoRA to `fc2`, the 64 x 64 layer that holds 4,160 of the model's 4,482 parameters.
`fc1` and `fc3` are left frozen and untouched.

### 🎯 Task — implement the LoRA branch

<details><summary>💡 <b>Hint 1</b> — what are we actually asking?</summary>

Read $BAx$ from the right: first $Ax$, which turns the 64-dimensional input into $r$ numbers, then $B$, which turns those $r$ numbers back into 64. Do not forget the scaling $\alpha/r$ in front. `h1` is a batch of row vectors, so a matrix acting on the right means multiplying by its transpose.

</details>

<details><summary>🔧 <b>Hint 2</b> — which variable, which function</summary>

`(self.alpha / self.r) * (h1 @ self.A.T) @ self.B.T`, with `self.A` of shape `(r, 64)` and `self.B` of shape `(64, r)`.

</details>

In [ ]:
class LoRAMLP(nn.Module):
    def __init__(self, state, r=4, alpha=8.0):
        super().__init__()
        self.backbone = MLP()
        self.backbone.load_state_dict(copy.deepcopy(state))
        for p in self.backbone.parameters():
            p.requires_grad = False                       # W stays exactly as pretrained

        self.r, self.alpha = r, alpha
        self.A = nn.Parameter(torch.randn(r, HIDDEN) * 0.01)   # (r, d_in)
        self.B = nn.Parameter(torch.zeros(HIDDEN, r))          # (d_out, r), starts at zero

    def forward(self, x):
        bb = self.backbone
        h1 = torch.relu(bb.fc1(x))
        frozen_part = bb.fc2(h1)                # W h1 + b, the pretrained layer
        lora_part   = ???                       # 🎯 the (alpha/r) B A h1 branch
        h2 = torch.relu(frozen_part + lora_part)
        return bb.fc3(h2)

seed_everything()
lora_model = LoRAMLP(PRETRAINED_STATE, r=4, alpha=8.0)
describe(lora_model)

512 trainable numbers, standing in for a 4,096-entry update. Check the starting point
first, then train.

In [ ]:
with torch.no_grad():
    same = torch.allclose(lora_model(X_tgt_test), backbone(X_tgt_test), atol=1e-6)
print("B = 0, so the LoRA model starts out as the pretrained model:", same)

lora_time = train(lora_model, X_tgt_train, y_tgt_train, epochs=200, lr=0.01)
print("trained in %.2f s · target accuracy %.1f%%"
      % (lora_time, 100 * accuracy(lora_model, X_tgt_test, y_tgt_test)))

## 4.2 · What the rank buys

Five ranks, same pretrained state, same everything else.

In [ ]:
ranks = (1, 2, 4, 8, 16)
lora_sweep = {}
for r in ranks:
    seed_everything()
    m = LoRAMLP(PRETRAINED_STATE, r=r, alpha=8.0)
    t = train(m, X_tgt_train, y_tgt_train, epochs=200, lr=0.01)
    lora_sweep[r] = {"trainable": count_trainable(m),
                     "target_acc": accuracy(m, X_tgt_test, y_tgt_test),
                     "time": t, "model": m}
    print("r=%-3d trainable %5d   target %.1f%%   %.2f s"
          % (r, lora_sweep[r]["trainable"], 100 * lora_sweep[r]["target_acc"], t))

pv.sweep_plot(list(ranks),
              [("trainable parameters", [lora_sweep[r]["trainable"] for r in ranks], pv.TRAIN),
               ("target accuracy (%)", [100 * lora_sweep[r]["target_acc"] for r in ranks],
                pv.GREEN, (80, 100)),
               ("training time (s)", [lora_sweep[r]["time"] for r in ranks], pv.GREY, (0, None))],
              xlabel="rank r", title="Rank: parameters, accuracy, runtime", logx=True)

pv.lora_widget(d_in=HIDDEN, d_out=HIDDEN, ranks=ranks, default=4,
               measured={r: {"trainable": v["trainable"], "target_acc": v["target_acc"]}
                         for r, v in lora_sweep.items()})

Read the middle panel honestly: on a two-dimensional toy problem the change the target
task needs is simple enough that even **r = 1** captures it, so accuracy is flat while the
parameter count multiplies by 16. That is a property of *this* problem, not a general result. The
lesson to carry away is the shape of the trade-off (rank is the capacity dial, and it costs
parameters linearly), not the flat line.

The third panel starts at zero on purpose. Sixteen times more trainable parameters cost nothing
measurable in training time, because the work is dominated by the frozen backbone that runs on
every step either way.

## 4.3 · Merging, and why it matters

The LoRA branch is a *linear* correction of a linear layer, so it can be folded into the weight
matrix once, after training:

$$W_{\text{merged}} = W + \frac{\alpha}{r} BA$$

After folding there is no branch left. The served model is a plain MLP with slightly different
numbers in `fc2`. Linearity is exactly what makes that fold possible: put a nonlinearity
anywhere inside the correction and there is no single matrix left to add.

Fold it and check numerically that nothing changed.

In [ ]:
lora_model = lora_sweep[4]["model"]
lora_time  = lora_sweep[4]["time"]

merged_model = fresh_backbone()
with torch.no_grad():
    delta_W = (lora_model.alpha / lora_model.r) * (lora_model.B @ lora_model.A)  # (64, 64)
    merged_model.fc2.weight += delta_W

lora_model.eval(); merged_model.eval()
with torch.no_grad():
    gap = (lora_model(X_tgt_test) - merged_model(X_tgt_test)).abs().max().item()

print("delta_W shape:", tuple(delta_W.shape), " built from", lora_model.A.numel(),
      "+", lora_model.B.numel(), "trained numbers")
print("largest difference between the two models' outputs: %.2e" % gap)
print("same accuracy: %.1f%% vs %.1f%%"
      % (100 * accuracy(lora_model, X_tgt_test, y_tgt_test),
         100 * accuracy(merged_model, X_tgt_test, y_tgt_test)))
assert gap < 1e-4, "merging must reproduce the branched model up to float32 tolerance"
print("✅ identical up to floating-point tolerance")

The point of that is not elegance, it is latency. Time all three.

In [ ]:
print("pretrained backbone : %.3f ms" % inference_ms(backbone, X_tgt_test))
print("LoRA, branch active : %.3f ms" % inference_ms(lora_model, X_tgt_test))
print("LoRA, merged        : %.3f ms" % inference_ms(merged_model, X_tgt_test))

> 🧷 Merging is a **deployment** choice, and it costs you the modularity: once folded, that
> copy of the model serves one task. The usual arrangement keeps the small `A`, `B` files as the
> stored artefact and merges into a copy at load time, per task.

Now the dashboard. The checkpoint holds `A` and `B`.

In [ ]:
lora_ckpt = checkpoint_bytes(task_checkpoint(lora_model), "lora_r4")
print("LoRA checkpoint:", lora_ckpt, "bytes")

lora_record = measure("LoRA", lora_model, lora_time, lora_ckpt,
                      changes_path=False, mergeable=True,
                      advantage="Small, mergeable task update with a capacity dial",
                      limitation="Rank and target layers are choices you have to make")
results.append(lora_record)

pv.dashboard(lora_record, baseline=BASELINE)

The `changes the execution path` tag says *no* because of the merge: the model you serve
can be exactly the shape of the original. The inference figure on the card is measured on the
**unmerged** model, which is what you pay if you keep the branch.

### 🧠 Quick check

In [ ]:
pv.mc_quiz("lora_count")

> ### What you just did
> You constrained the *update* to a low-rank form while leaving the pretrained weight matrix
> full-rank and frozen, and then made the update disappear into the weights. One method left, and
> it attacks the same problem from the opposite side: keep the update dense, but make almost all
> of it zero.

---
# Part 5 — Diff Pruning: a sparse difference  ·  ~7 min

The third idea. Write the adapted model as the base model plus a difference:

$$\theta_{\text{task}} = \theta_{\text{base}} + \Delta\theta$$

and push almost every entry of $\Delta\theta$ to **exactly zero**. What you store per task is then
a short list: which parameters changed, and by how much.

The version implemented here keeps the mechanism visible and skips the delicate optimisation of
the original paper:

1. create a trainable difference tensor for **every** base parameter;
2. train with an **L1 penalty** on the differences, which pulls small ones towards zero;
3. **threshold** after training: anything below a cutoff becomes exactly zero;
4. report the dense training-state size and the sparse checkpoint size **separately**.

> ⚠️ Step 1 is the catch, and it is the reason this method is in the notebook. While training,
> there is a difference variable (plus its gradient, plus its Adam moments) for every parameter
> being adapted. The small file at the end says nothing about that.

In [ ]:
import torch.nn.functional as F

class DiffPruned(nn.Module):
    def __init__(self, state):
        super().__init__()
        self.backbone = MLP()
        self.backbone.load_state_dict(copy.deepcopy(state))
        for p in self.backbone.parameters():
            p.requires_grad = False
        # one trainable difference per base parameter; ParameterDict keys cannot hold dots
        self.delta = nn.ParameterDict({
            name.replace(".", "_"): nn.Parameter(torch.zeros_like(p))
            for name, p in self.backbone.named_parameters()})

    def effective(self, name):
        '''base + difference, for one parameter tensor'''
        base = dict(self.backbone.named_parameters())[name]
        return base + self.delta[name.replace(".", "_")]

    def forward(self, x):
        h1 = torch.relu(F.linear(x, self.effective("fc1.weight"), self.effective("fc1.bias")))
        h2 = torch.relu(F.linear(h1, self.effective("fc2.weight"), self.effective("fc2.bias")))
        return F.linear(h2, self.effective("fc3.weight"), self.effective("fc3.bias"))

def train_with_l1(model, X, y, epochs=200, lr=0.01, l1=0.01):
    '''The usual loop, plus a penalty on the size of the differences.'''
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimiser = torch.optim.Adam(trainable, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    start = time.perf_counter()
    for _ in range(epochs):
        optimiser.zero_grad()
        penalty = sum(d.abs().sum() for d in model.delta.values())
        loss = loss_fn(model(X), y) + l1 * penalty
        loss.backward()
        optimiser.step()
    return time.perf_counter() - start

seed_everything()
diff_model = DiffPruned(PRETRAINED_STATE)
diff_time = train_with_l1(diff_model, X_tgt_train, y_tgt_train, epochs=200, lr=0.01, l1=0.01)

print("trainable during training:", count_trainable(diff_model),
      "  <- one per base parameter, the same as full fine-tuning")
print("trained in %.2f s · target accuracy %.1f%%"
      % (diff_time, 100 * accuracy(diff_model, X_tgt_test, y_tgt_test)))

## 5.1 · Threshold, then count

Keep a copy of the dense differences first, so we can look at the before and after.

In [ ]:
delta_before = {name: d.detach().clone() for name, d in diff_model.delta.items()}
big = torch.cat([d.flatten().abs() for d in delta_before.values()])
print("differences above 0.05 :", int((big > 0.05).sum()), "of", big.numel())
print("largest difference     : %.3f" % big.max().item())



The L1 penalty makes most differences small. It does not make them zero, so we cut them.

### 🎯 Task — apply the threshold and count what survives

<details><summary>💡 <b>Hint 1</b> — what are we actually asking?</summary>

Two things per tensor: set every entry whose absolute value is below the cutoff to exactly zero, then count how many entries are still nonzero. Do it inside `torch.no_grad()`, because this is surgery on the trained values, not a training step.

</details>

<details><summary>🔧 <b>Hint 2</b> — which variable, which function</summary>

`d[d.abs() < threshold] = 0.0` edits the tensor in place; `int((d != 0).sum())` counts what is left.

</details>

In [ ]:
THRESHOLD = 0.05

nonzero = 0
with torch.no_grad():
    for name, d in diff_model.delta.items():
        ???                          # 🎯 zero out every difference smaller than THRESHOLD
        nonzero += ???               # 🎯 count the differences that survived

dense = count_trainable(diff_model)
print("differences trained :", dense)
print("still nonzero       :", nonzero, "(%.1f%% of them)" % (100 * nonzero / dense))
print("target accuracy after thresholding: %.1f%%"
      % (100 * accuracy(diff_model, X_tgt_test, y_tgt_test)))

Where did the survivors end up? Not spread evenly.

In [ ]:
for name, d in diff_model.delta.items():
    print("%-12s %4d of %5d survived" % (name, int((d != 0).sum()), d.numel()))

Almost the whole update sits in the **first** layer, and `fc2` (4,096 of the model's 4,482
parameters) keeps a handful of entries. That fits the task: the target moons are the source moons
rotated, and a rotation of the input is something the first layer can absorb. Nothing forced that
outcome, the L1 penalty simply found it cheaper.

Here is the entire update as a grid, one cell per parameter, in parameter order. Colour is the
size of the difference; white cells are padding at the end.

In [ ]:
flat_before = torch.cat([d.flatten() for d in delta_before.values()]).numpy()
pv.diff_grid(flat_before, threshold=THRESHOLD)

## 5.2 · What a sparse checkpoint actually costs

A sparse update is not just its values. Every value needs to say *where* it goes, as an index or
as a mask. Both go in the file.

In [ ]:
flat = torch.cat([d.detach().flatten() for d in diff_model.delta.values()])
positions = torch.nonzero(flat, as_tuple=False).flatten().to(torch.int32)
values    = flat[positions.long()]

sparse_state = {"positions": positions, "values": values}   # where, and how much
diff_ckpt = checkpoint_bytes(sparse_state, "diff_pruning")
estimated = nonzero * (4 + 4)          # 4 bytes per value, 4 per position

print("nonzero values     :", nonzero)
print("estimated payload  : %d bytes  (value + position, 4 bytes each)" % estimated)
print("actual file        : %d bytes  (torch adds dtypes, shapes and pickle framing)" % diff_ckpt)
print("BitFit             : %d bytes" % bitfit_ckpt)
print("full fine-tuning   : %d bytes" % full_ckpt)

At this size the file is mostly wrapping: 592 bytes of payload inside a file of a couple
of kilobytes. That ratio is a property of a 4,482-parameter model, not of the method. Framing
costs are roughly fixed, while the payload grows with the number of parameters you adapt.

For inference, the difference is folded back into the base weights: the served model is a plain
MLP again, the same shape as the original. That is what gets measured below.

In [ ]:
diff_merged = fresh_backbone()
with torch.no_grad():
    for name, p in diff_merged.named_parameters():
        p += diff_model.delta[name.replace(".", "_")]

# what training cost: the base weights AND a dense difference for each of them
diff_train_state = training_state_bytes(count_parameters(diff_model), count_trainable(diff_model))

diff_record = measure("Diff Pruning", diff_merged, diff_time, diff_ckpt,
                      trainable_override=dense, train_state_override=diff_train_state,
                      changes_path=False, mergeable=True,
                      advantage="A genuinely sparse task update, saved as a short list",
                      limitation="Dense difference variables while training, and a sparse "
                                 "format to maintain afterwards")
results.append(diff_record)

pv.dashboard(diff_record, baseline=BASELINE)

Read that card carefully, because it is the one that breaks the pattern. What this task
owns is 74 numbers and their positions, a payload of well under a kilobyte. And the training-state
estimate is the **largest of any method, including full fine-tuning**, because training carried
the base weights plus a difference, a gradient and two Adam moments for every one of them.

> ### What you store after training and what you had to allocate during training are two
> ### different quantities. Diff Pruning is the method that makes that impossible to ignore.

### 🧠 Quick check

In [ ]:
pv.mc_quiz("diff_memory")

---
# Part 6 — All of it at once, and how to choose  ·  ~7 min

Four methods, one pretrained state, one target task, one training budget. Here is everything the
notebook measured.

In [ ]:
pv.results_table(results,
                 extra="Every row started from the same PRETRAINED_STATE, used the same "
                       "target split, the same optimiser (Adam, lr 0.01), the same 200 epochs "
                       "and full batches. The classification head weight fc3.weight was frozen "
                       "for every PEFT method; BitFit moves its bias, because a bias is what "
                       "BitFit trains.")

## 6.1 · The three axes that disagree

Each group of bars is a percentage of full fine-tuning. Note that they do not move together:
that is the whole finding.

In [ ]:
pv.normalised_bars(results, baseline=BASELINE)

Accuracy against the number of parameters you had to train. Cheap is to the left, accurate
is at the top.

In [ ]:
pv.pareto(results)

And the number Part 0 opened with, now with measured checkpoint sizes instead of assumed
ones. The backbone is stored once for every method that shares it.

In [ ]:
pv.storage_projection(results, backbone_bytes=full_ckpt, tasks=(1, 5, 20, 100))

Finally, what all of that bought on the actual task.

In [ ]:
pv.boundaries([("pretrained", predict_fn(backbone)),
               ("full fine-tuning", predict_fn(full_model)),
               ("BitFit", predict_fn(bitfit_model)),
               ("LoRA (r=4)", predict_fn(lora_model)),
               ("Diff Pruning", predict_fn(diff_merged))],
              X_tgt, y_tgt, title="Target task, every method", ncols=3)

## 6.2 · What happened to the task it was pretrained on

Every PEFT method here kept the pretrained weights frozen, and every one of them still moved the
model's behaviour on the source task. Compare the source-accuracy column against where it
started.

In [ ]:
print("source accuracy before adapting: %.1f%%\n" % (100 * acc_src_before))
for r in results:
    print("%-16s target %.1f%%   source %.1f%%   (%+.1f points)"
          % (r["method"], 100 * r["target_acc"], 100 * r["source_acc"],
             100 * (r["source_acc"] - acc_src_before)))

Frozen weights preserve the pretrained model **as a set of values**, which is why you can
always recover it by detaching the module. They do not preserve its **behaviour** while the
module is attached: the computation running is a different one. Reduced interference is a
plausible side effect of freezing, not a guarantee, and it is not the reason a cluster operator
reaches for PEFT.

### 🧠 Quick checks

In [ ]:
pv.mc_quiz("speedup")

In [ ]:
pv.mc_quiz("forgetting")

In [ ]:
pv.true_false_quiz("systems")

## 6.3 · 🎯 Final task — choose, and defend the choice

Three situations. For each one, pick a method and write **two or three sentences** justifying it
from the numbers in your table. There is no single right answer, and the argument is what counts:
name the column you are optimising and the one you are giving up.

Click a scenario once you have written your answer.

In [ ]:
pv.choose_method_panel()

---
# Part 7 — The same mechanisms, on a real model  ·  ~2 min

Everything in this notebook happened on three linear layers. Large pretrained models contain many
more of them, arranged in repeated blocks, and that is essentially the only difference:

- **BitFit** selects the bias vectors that are already there, wherever they are.
- **LoRA** attaches a low-rank update to chosen linear transformations, and is merged for serving
  when latency matters.
- **Diff Pruning** learns a sparse difference from the shared base parameters.

Libraries automate the mechanical part: finding the layers, inserting the modules, flipping
`requires_grad`, saving only the small state. What they do not change is any of the trade-offs you
just measured. A frozen backbone is still loaded and still executed. A module still adds work to
the forward pass unless it can be merged. A sparse checkpoint still says nothing about training
memory. The numbers get bigger; the columns of your table stay the same.

In [ ]:
pv.takeaway()

---
# 🎓 Wrap-up — the four methods, side by side

| Method | What is trained | Main strength | Main limitation |
|---|---|---|---|
| **Full fine-tuning** | every base parameter | maximum freedom to adapt | full training state, full checkpoint per task |
| **BitFit** | existing bias vectors | nothing added, tiny checkpoint | little capacity to reshape a layer |
| **LoRA** | two thin factors of the update | small, mergeable, capacity dial | rank and layer choice matter |
| **Diff Pruning** | a sparse difference from the base | very small sparse update | dense training state, sparse plumbing |

Check that against your own dashboards rather than trusting the row: the numbers came out of your
run, and one toy dataset is not a ranking.

**The one-line takeaway:** *PEFT is about what has to be trained and stored per task, not about
making the pretrained model smaller.*

---
## 🏁 Final boss — clear the notebook

Everything in one place, one statement at a time: **frozen versus trainable, training state
versus checkpoint, BitFit, LoRA, merging, Diff Pruning.** The rules: **3 lives**,
**10 seconds** per statement, a wrong answer *or* a timeout costs a life. Reach **10 correct** to
pass. 🍀

In [ ]:
pv.flash_quiz()

---
## Where this leaves you

You started with a storage bill for a hundred tasks and ended with four methods measured on the
same model, the same split and the same budget. Along the way: freezing changes the backward side
of the ledger and leaves the forward side alone; rank is a capacity dial that costs parameters;
a low-rank update can be folded into the weights it corrects; and a tiny checkpoint can come out
of an expensive training run.

> ### The pretrained backbone does not get smaller. What PEFT changes is the cost of task number 101.